<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin: 20px; height: 55px">

## Neural Networks for NLP


_Author:_ Tim Book (and technically also Francois Chollet, mostly)

In [1]:
import os
import shutil
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

# Get IMDB Review Data
Data are IMDB reviews (text) with sentiment as target (positive or negative). The data comes from Stanford, where they provide us with training data, testing data, and unsupervised data which we delete. We also will pull 20% of the training data into a new "validation" folder.

In [2]:
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz
!rm -r aclImdb/train/unsup

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  8906k      0  0:00:09  0:00:09 --:--:-- 15.3M


In [3]:
train_dir = f'aclImdb/train'
val_dir = f'aclImdb/val'

if not os.path.exists(val_dir):
  os.mkdir(val_dir)

for cat in ['pos', 'neg']:
  train_files = os.listdir(os.path.join(train_dir, cat))
  n_train = len(train_files)

  # Separate 20% of training for validation
  np.random.seed(123)
  val_files = np.random.choice(train_files, size=int(0.2*n_train), replace=False)

  # Create val directory
  if not os.path.exists(os.path.join(val_dir, cat)):
    os.mkdir(os.path.join(val_dir, cat))

  # Move files from train to val
  for f in val_files:
    shutil.move(f'{train_dir}/{cat}/{f}', f'{val_dir}/{cat}/{f}')

# Get GloVe Word Vectors
The `wget` command pulls data from the web. We unzip this folder and read in word vectors as Python objects.

In [4]:
# UNCOMMENT BELOW AND RUN ASAP!
# This cell can take up to 5 minutes to run, as it's downloading and unzipping
# a large file. Once run, please re-comment it so you don't accidentally run
# it twice!

!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip

--2023-03-12 22:22:44--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2023-03-12 22:22:44--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2023-03-12 22:22:45--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

# Create Keras Datasets
These are special datasets that pull data from a specific folder structure which we have curated.

In [5]:
batch_size = 32
train_ds = keras.utils.text_dataset_from_directory('aclImdb/train', batch_size=batch_size)
val_ds = keras.utils.text_dataset_from_directory('aclImdb/val', batch_size=batch_size)
test_ds = keras.utils.text_dataset_from_directory('aclImdb/test', batch_size=batch_size)

# Get dataset of only text (no labels)
text_only_train_ds = train_ds.map(lambda x, y: x)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.


Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089


Found 25000 files belonging to 2 classes.


In [6]:
max_length = 600
max_tokens = 20_000

# Create an object that can process strings into integet incodings based on our
# trainind data vocabulary.
text_vectorization = keras.layers.TextVectorization(
    max_tokens=max_tokens, 
    output_mode='int', 
    output_sequence_length=max_length
)
text_vectorization.adapt(text_only_train_ds)

In [7]:
text_vectorization('the cat sat on the mat')[:8]

<tf.Tensor: shape=(8,), dtype=int64, numpy=array([    2,  1112,  1819,    21,     2, 14619,     0,     0])>

In [8]:
text_vectorization('the xyz sat on the mat')[:8]

<tf.Tensor: shape=(8,), dtype=int64, numpy=array([    2,     1,  1819,    21,     2, 14619,     0,     0])>

In [9]:
# Create datasets that turn our text input into integer encodings
int_mapper = lambda x, y: (text_vectorization(x), y)

int_train_ds = train_ds.map(int_mapper, num_parallel_calls=4)
int_val_ds = val_ds.map(int_mapper, num_parallel_calls=4)
int_test_ds = test_ds.map(int_mapper, num_parallel_calls=4)

In [10]:
inputs = keras.Input(shape=(None,))
embedded = layers.Embedding(
    input_dim=max_tokens,  # Maximum of 600 tokens per sequence
    output_dim=256,        # Output of 256-length word vecs
    mask_zero=True         # This tells our model to skip over 0-padding tokens
)(inputs)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs, outputs)
model.compile(optimizer='rmsprop', loss='bce', metrics=['acc'])
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, None)]            0         
                                                                 
 embedding (Embedding)       (None, None, 256)         5120000   
                                                                 
 bidirectional (Bidirectiona  (None, 64)               73984     
 l)                                                              
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense (Dense)               (None, 1)                 65        
                                                                 
Total params: 5,194,049
Trainable params: 5,194,049
Non-trainable params: 0
___________________________________________________

In [11]:
# DO NOT RUN DURING CLASS!!!
# This cell can take between 5 and 10 minutes to run - run on your own time.
# Spoiler alert: Testing accuracy of about 87%
# model.fit(
#     int_train_ds,
#     validation_data=int_val_ds,
#     epochs=10
# )

# model.evaluate(int_test_ds)

Epoch 1/10
625/625 [==============================] - 100s 139ms/step - loss: 0.4583 - acc: 0.7800 - val_loss: 0.3202 - val_acc: 0.8706
Epoch 2/10
625/625 [==============================] - 50s 79ms/step - loss: 0.2798 - acc: 0.8895 - val_loss: 0.3063 - val_acc: 0.8736
Epoch 3/10
625/625 [==============================] - 42s 68ms/step - loss: 0.2157 - acc: 0.9177 - val_loss: 0.2956 - val_acc: 0.8824
Epoch 4/10
625/625 [==============================] - 35s 56ms/step - loss: 0.1647 - acc: 0.9406 - val_loss: 0.3187 - val_acc: 0.8890
Epoch 5/10
625/625 [==============================] - 34s 54ms/step - loss: 0.1306 - acc: 0.9535 - val_loss: 0.3608 - val_acc: 0.8742
Epoch 6/10
625/625 [==============================] - 32s 52ms/step - loss: 0.0979 - acc: 0.9668 - val_loss: 0.4213 - val_acc: 0.8712
Epoch 7/10
625/625 [==============================] - 33s 53ms/step - loss: 0.0768 - acc: 0.9736 - val_loss: 0.4418 - val_acc: 0.8840
Epoch 8/10
625/625 [==============================] - 31s 49

[0.5721599459648132, 0.8683199882507324]

# Fitting the same model with GloVe's pretrained embeddings

In [12]:
embeddings_ix = {}

with open('./glove.6B.100d.txt', 'r') as f:
  for line in f:
    word, coefs = line.split(maxsplit=1)
    coefs = np.fromstring(coefs, 'f', sep=' ')
    embeddings_ix[word] = coefs

In [13]:
embeddings_ix['cat'][:4]

array([ 0.23088,  0.28283,  0.6318 , -0.59411], dtype=float32)

In [14]:
embedding_dim = 100

vocab = text_vectorization.get_vocabulary()
word_ix = dict(enumerate(vocab))
word_ix = {word: i for i, word in word_ix.items()}

In [15]:
# 0 and 1 occupy special places in this architecture!
print(word_ix[''])
print(word_ix['[UNK]'])
print(word_ix['cat'])

0
1
1112


In [16]:
# Create a huge embedding matrix based on GloVe - no need to train it ourself!
# 1 row = 1 word vector
embedding_mx = np.zeros((max_tokens, embedding_dim))

for word, i in word_ix.items():
  if i < max_tokens:
    embed_vector = embeddings_ix.get(word)
  
  if embed_vector is not None:
    embedding_mx[i, :] = embed_vector

In [17]:
embedding_mx[1112, :5]

array([ 0.23088001,  0.28283   ,  0.6318    , -0.59411001, -0.58599001])

In [18]:
embeddings_ix['cat'][:5]

array([ 0.23088,  0.28283,  0.6318 , -0.59411, -0.58599], dtype=float32)

In [19]:
# Create a "frozen" embedding layer
glove_layer = layers.Embedding(
    max_tokens,
    embedding_dim,
    embeddings_initializer=keras.initializers.Constant(embedding_mx),
    trainable=False,
    mask_zero=True
)

In [20]:
inputs = keras.Input(shape=(None,))
embedded = glove_layer(inputs)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer='rmsprop', loss='bce', metrics=['acc'])
model.summary()

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, None)]            0         
                                                                 
 embedding_1 (Embedding)     (None, None, 100)         2000000   
                                                                 
 bidirectional_1 (Bidirectio  (None, 64)               34048     
 nal)                                                            
                                                                 
 dropout_1 (Dropout)         (None, 64)                0         
                                                                 
 dense_1 (Dense)             (None, 1)                 65        
                                                                 
Total params: 2,034,113
Trainable params: 34,113
Non-trainable params: 2,000,000
____________________________________________

In [21]:
model.fit(
    int_train_ds,
    validation_data=int_val_ds,
    epochs=10
)

Epoch 1/10
625/625 [==============================] - 42s 54ms/step - loss: 0.5850 - acc: 0.6786 - val_loss: 0.5166 - val_acc: 0.7392
Epoch 2/10
625/625 [==============================] - 29s 47ms/step - loss: 0.4631 - acc: 0.7868 - val_loss: 0.4168 - val_acc: 0.8096
Epoch 3/10
625/625 [==============================] - 30s 48ms/step - loss: 0.4086 - acc: 0.8189 - val_loss: 0.4049 - val_acc: 0.8146
Epoch 4/10
625/625 [==============================] - 29s 47ms/step - loss: 0.3716 - acc: 0.8365 - val_loss: 0.3654 - val_acc: 0.8386
Epoch 5/10
625/625 [==============================] - 29s 47ms/step - loss: 0.3457 - acc: 0.8522 - val_loss: 0.3583 - val_acc: 0.8466
Epoch 6/10
625/625 [==============================] - 30s 48ms/step - loss: 0.3197 - acc: 0.8659 - val_loss: 0.3252 - val_acc: 0.8622
Epoch 7/10
625/625 [==============================] - 30s 47ms/step - loss: 0.3020 - acc: 0.8735 - val_loss: 0.3291 - val_acc: 0.8612
Epoch 8/10
625/625 [==============================] - 30s 48ms

In [22]:
model.evaluate(int_test_ds)

782/782 [==============================] - 16s 20ms/step - loss: 0.3101 - acc: 0.8682


[0.31005859375, 0.8682399988174438]